# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_token")

In [3]:
from huggingface_hub import login

login(HF_TOKEN)

In [6]:
from huggingface_hub import hf_hub_download
dim_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN,
)

fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [7]:
import pandas as pd

dim_df = pd.read_parquet(dim_path)
fact_df = pd.read_parquet(fact_path)

In [8]:
print(dim_df.shape)
print(fact_df.shape)

dim_df.head()

(519606, 26)
(9841378, 30)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [ ]:
fact_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

The warehouse contains two related tables:

- **dim_content:** One row represents one unique content item (page/article) and stores its metadata.
- **fact_content_daily_performance:** One row represents the performance metrics of one content item for one reporting date.

### Time window

The fact table stores **daily** performance snapshots rather than pre-aggregated 30-day or 90-day metrics.

This analysis uses the available partition covering **March 2026** (`report_date = 2026-03-*`), while the warehouse itself contains monthly partitions through **June 2026**.

The statements below verify these assumptions.

In [9]:
# Shape of both tables
print("dim_content:", dim_df.shape)
print("fact_content_daily_performance:", fact_df.shape)

# Date range
print("\nDate range:")
print(fact_df["report_date"].min())
print(fact_df["report_date"].max())

# Check uniqueness of the grain
grain = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

print("\nTotal rows:", len(fact_df))
print("Unique grain combinations:", fact_df[grain].drop_duplicates().shape[0])

dim_content: (519606, 26)
fact_content_daily_performance: (9841378, 30)

Date range:
2026-03-01
2026-03-31

Total rows: 9841378
Unique grain combinations: 9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

1. `word_count`
   - Content length measured in words.
   - Available when the content is created or updated.

2. `content_type`
   - The type of content, such as a keyword article.
   - Available when the content is created.

3. `content_created_date`
   - The date the content was created; can be transformed into content age.
   - Available when the content is created.

4. `keyword_token_count`
   - Number of tokens in the target keyword.
   - Available when the keyword is created.

5. `url_char_count`
   - Number of characters in the page URL.
   - Available when the URL is created.

### Label

A future content-performance outcome derived from Google Search Console data.
The prediction horizon will be defined during model framing.

### Context

- `report_date` — identifies the reporting period.
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content item.
- `keyword_hash_id` — identifies the keyword.
- `url_hash_id` — identifies the URL.

### Excluded

- `provider_used` — generation metadata, not a content-performance feature.
- `model_used` — generation metadata, not a content-performance feature.
- `is_deleted` — used for filtering/data quality rather than prediction.
- `is_published` — status information; handled as a data-quality/filtering field.
- GSC performance fields from the prediction period — excluded from features when they would only become available after the prediction point.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Query 1: Verify row count and grain

print("Total rows:", len(fact_df))

unique_grain = fact_df[
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates()

print("Unique grain combinations:", len(unique_grain))

print(
    "Duplicate grain combinations:",
    len(fact_df) - len(unique_grain)
)

Total rows: 9841378
Unique grain combinations: 9841378
Duplicate grain combinations: 0


In [11]:
# Query 2: Verify availability using an explicit TRUE check

print("GSC available (TRUE):")
print(fact_df["gsc_data_available"].eq(True).sum())

print("\nGA4 available (TRUE):")
print(fact_df["ga4_data_available"].eq(True).sum())

GSC available (TRUE):
3611061

GA4 available (TRUE):
413966


In [12]:
# Query 3: Verify reporting window

print("Earliest report date:", fact_df["report_date"].min())
print("Latest report date:", fact_df["report_date"].max())

print("\nNumber of reporting dates:",
      fact_df["report_date"].nunique())

Earliest report date: 2026-03-01
Latest report date: 2026-03-31

Number of reporting dates: 31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitation

A major limitation of this data is uneven availability of analytics sources. In the March 2026 verification slice, GSC data is available for only a subset of rows, while GA4 data is available for an even smaller subset. Therefore, features based on these sources cannot be assumed to be available for every content record.

This matters because a model using GSC or GA4 features may have a smaller usable training population or require explicit missing-value handling. The March 2026 slice is used here for contract verification; model training will use the full available dataset.

In [13]:
total_rows = len(fact_df)

gsc_available = fact_df["gsc_data_available"].eq(True).sum()
ga4_available = fact_df["ga4_data_available"].eq(True).sum()

print(f"Total rows: {total_rows:,}")
print(f"GSC available: {gsc_available:,} ({gsc_available / total_rows:.1%})")
print(f"GA4 available: {ga4_available:,} ({ga4_available / total_rows:.1%})")

Total rows: 9,841,378
GSC available: 3,611,061 (36.7%)
GA4 available: 413,966 (4.2%)


### Data limits

This slice has uneven data availability across measurement sources. GSC data is available for 36.7% of rows, while GA4 data is available for only 4.2% of rows. Therefore, GSC and GA4 cannot be assumed to be available for every content-performance record, and models using these fields must account for missing availability.

This notebook verifies a March 2026 slice. The later training workflow will use the full available dataset rather than relying only on this one-month slice.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.